# Ingest Races.csv file Assignment


## Assignment
- create a DF loading the races.csv file in the raw container
- rename columns to (race_id,race_year,circuit_id)
- create a new column race_timestampt from the columns (date, time)
- add a new columns ingestion_timestampt
- save the file in parquet file in the processed container.
- verify the schema of the partquet file

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType,DateType
from pyspark.sql.functions import current_timestamp,to_date,current_date,lit,to_timestamp,concat,col


In [0]:
races_df = spark.read.csv("abfs://raw@f1datalakefp.dfs.core.windows.net/races.csv",header=True)
#display(races_df)

In [0]:
races_schema = StructType(fields=[StructField("race_id", IntegerType(), False),
                                     StructField("race_year", IntegerType(), True),
                                     StructField("round", IntegerType(), True),
                                     StructField("circuit_id", IntegerType(), True),
                                     StructField("name", StringType(), True),
                                     StructField("date", DateType(), True),
                                     StructField("time", StringType(), True)
                                    ])
races_df = spark.read.schema(races_schema).csv("abfs://raw@f1datalakefp.dfs.core.windows.net/races.csv",header=True)

In [0]:
races_df = races_df.withColumn('ingestion_timestamp', current_timestamp()).withColumn('race_timestamp', to_timestamp(concat(col('date'),lit(' '),col('time')), 'yyyy-MM-dd HH:mm:ss'))

In [0]:
races_df=races_df.drop('time').drop('date')

In [0]:
#display(races_df)

## Write DF into parquet file

In [0]:
races_df.write.mode("overwrite").partitionBy("race_year").parquet("abfs://processed@f1datalakefp.dfs.core.windows.net/races")

In [0]:
df=spark.read.parquet("abfs://processed@f1datalakefp.dfs.core.windows.net/races")
df.printSchema()